In [1]:
import os
import pandas as pd

DOWNLOAD_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_images/extra_images'
CSV_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_csvs'

print("📁 Step 1: Scanning physical species folders...")
current_species_folders = [f for f in os.listdir(DOWNLOAD_DIR) if not f.startswith('.')]

all_records = []
for folder in current_species_folders:
    species_name = folder.replace('_', ' ').strip()
    folder_path = os.path.join(DOWNLOAD_DIR, folder)
    images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for img in images:
        all_records.append({
            'species': species_name,
            'local_path': os.path.join(folder_path, img)
        })

combined_df = pd.DataFrame(all_records)
print(f"📊 Total raw images found on disk: {len(combined_df)} across {combined_df['species'].nunique()} species.")

print("\n🌲 Step 2: Extracting California checklist from metadata sheets...")
target_csvs = ['mammal_class.csv', 'amphibia_class.csv', 'reptilia_class.csv', 'aves_class.csv']
ca_species_set = set()

for csv_file in target_csvs:
    csv_path = os.path.join(CSV_DIR, csv_file)
    if os.path.exists(csv_path):
        df_csv = pd.read_csv(csv_path, usecols=['scientific_name', 'place_admin1_name', 'place_state_name', 'place_guess'], low_memory=False)
        
        ca_mask = (
            (df_csv['place_admin1_name'].astype(str).str.strip() == 'California') |
            (df_csv['place_state_name'].astype(str).str.strip().str.lower() == 'california') |
            (df_csv['place_guess'].astype(str).str.lower().str.contains('california', na=False))
        )
        for s in df_csv[ca_mask]['scientific_name'].dropna().unique():
            ca_species_set.add(str(s).strip().lower())

print(f"✅ Metadata checklist built: {len(ca_species_set)} species verified in California.")

print("\n⚡ Step 3: Filtering disk dataframe for California locals...")
is_ca_native = combined_df['species'].str.lower().str.strip().isin(ca_species_set)
combined_df = combined_df[is_ca_native].reset_index(drop=True)

print("\n✂️ Step 4: SHAVING TARGETS DOWN TO TOP 30 HIGHEST VOLUME SPECIES...")
# Count images per species and isolate the top 30 most data-heavy targets
species_counts = combined_df['species'].value_counts()
top_30_species = species_counts.head(30).index

# Drop everything else out of the dataframe
combined_df = combined_df[combined_df['species'].isin(top_30_species)].reset_index(drop=True)

# Map targets dynamically based on the finalized top-30 list
unique_ca_species = sorted(combined_df['species'].unique())
species_to_id = {species: idx for idx, species in enumerate(unique_ca_species)}
combined_df['label'] = combined_df['species'].map(species_to_id)

print("-" * 60)
print(f"🌲 SUCCESS: Scoped Dataset Formed for a Fast 70%+ Run!")
print(f"Total California Species Kept: {len(unique_ca_species)} (Targeted Top 30)")
print(f"Total Verified Images Loaded:   {len(combined_df)}")
print(f"PyTorch Label Range:           0 to {combined_df['label'].max()}")

📁 Step 1: Scanning physical species folders...
📊 Total raw images found on disk: 35014 across 300 species.

🌲 Step 2: Extracting California checklist from metadata sheets...
✅ Metadata checklist built: 323 species verified in California.

⚡ Step 3: Filtering disk dataframe for California locals...

✂️ Step 4: SHAVING TARGETS DOWN TO TOP 30 HIGHEST VOLUME SPECIES...
------------------------------------------------------------
🌲 SUCCESS: Scoped Dataset Formed for a Fast 70%+ Run!
Total California Species Kept: 30 (Targeted Top 30)
Total Verified Images Loaded:   3937
PyTorch Label Range:           0 to 29


In [2]:
# import os
# import pandas as pd

# DOWNLOAD_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_images/extra_images'
# CSV_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_csvs'

# print("📁 Step 1: Scanning physical folders with a 100-image cap...")
# current_species_folders = [f for f in os.listdir(DOWNLOAD_DIR) if not f.startswith('.')]

# all_records = []
# for folder in current_species_folders:
#     species_name = folder.replace('_', ' ').strip()
#     folder_path = os.path.join(DOWNLOAD_DIR, folder)
#     images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
#     capped_images = images[:100]
#     for img in capped_images:
#         all_records.append({
#             'species': species_name,
#             'local_path': os.path.join(folder_path, img)
#         })

# combined_df = pd.DataFrame(all_records)

# print("🌲 Step 2: Extracting California checklist and filtering for vertebrates...")
# target_csvs = {
#     'Mammalia':  'mammal_class.csv',
#     'Amphibia':  'amphibia_class.csv',
#     'Reptilia':  'reptilia_class.csv',
#     'Aves':      'aves_class.csv',
# }

# ca_species_set = set()
# species_to_class_map = {}

# for class_name, csv_file in target_csvs.items():
#     csv_path = os.path.join(CSV_DIR, csv_file)
#     if os.path.exists(csv_path):
#         df_csv = pd.read_csv(csv_path, usecols=['scientific_name', 'place_admin1_name', 'place_state_name', 'place_guess'], low_memory=False)
        
#         for s in df_csv['scientific_name'].dropna().unique():
#             species_to_class_map[str(s).strip().lower()] = class_name
            
#         ca_mask = (
#             (df_csv['place_admin1_name'].astype(str).str.strip() == 'California') |
#             (df_csv['place_state_name'].astype(str).str.strip().str.lower() == 'california') |
#             (df_csv['place_guess'].astype(str).str.lower().str.contains('california', na=False))
#         )
#         for s in df_csv[ca_mask]['scientific_name'].dropna().unique():
#             ca_species_set.add(str(s).strip().lower())

# is_ca_native = combined_df['species'].str.lower().str.strip().isin(ca_species_set)
# combined_df = combined_df[is_ca_native].reset_index(drop=True)

# combined_df['taxonomic_class'] = combined_df['species'].str.lower().str.strip().map(species_to_class_map)
# combined_df = combined_df[combined_df['taxonomic_class'].notna()].reset_index(drop=True)

# print("✂️ Step 3: Enforcing minimum image floor (>= 60 images per species)...")
# species_counts = combined_df['species'].value_counts()
# high_volume_species = species_counts[species_counts >= 60].index
# combined_df = combined_df[combined_df['species'].isin(high_volume_species)].reset_index(drop=True)

# unique_ca_species = sorted(combined_df['species'].unique())
# species_to_id = {species: idx for idx, species in enumerate(unique_ca_species)}
# combined_df['label'] = combined_df['species'].map(species_to_id)

# print("-" * 60)
# print(f"🌲 SUCCESS: Clean Dataset Locked In!")
# print(f"Unique Vertebrate Species Left: {len(unique_ca_species)}")
# print(f"Total Combined Images:         {len(combined_df)}")
# print(f"PyTorch Label Range:           0 to {combined_df['label'].max()}")
# print("-" * 60)

In [3]:
# =====================================================================
# TAXONOMIC CLASS SPLIT BREAKDOWN
# =====================================================================
import os
import pandas as pd

CSV_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_csvs'

# 1. Map CSV filenames to their true taxonomic Class name
target_csvs = {
    'Mammalia':  'mammal_class.csv',
    'Amphibia':  'amphibia_class.csv',
    'Reptilia':  'reptilia_class.csv',
    'Aves':      'aves_class.csv',
    'Insecta':   'insecta_class.csv',
    'Arachnida': 'arachnida_class.csv',
}

# 2. Build a quick global dictionary mapping: species_name -> taxonomic_class
species_to_class_map = {}
for class_name, csv_file in target_csvs.items():
    csv_path = os.path.join(CSV_DIR, csv_file)
    if os.path.exists(csv_path):
        df_csv = pd.read_csv(csv_path, usecols=['scientific_name'], low_memory=False)
        for s in df_csv['scientific_name'].dropna().unique():
            species_to_class_map[str(s).strip().lower()] = class_name

# 3. Add the Class mapping onto our existing dataframe copy
df_distribution = combined_df.copy()
df_distribution['taxonomic_class'] = df_distribution['species'].str.lower().str.strip().map(species_to_class_map)
# Fallback for any outliers that might be sitting inside a general phylum/chordata dump
df_distribution['taxonomic_class'] = df_distribution['taxonomic_class'].fillna('Other / Chordata')

# 4. Print High-Level Macro Class Stats
print("📊 MACRO CLASS DISTRIBUTION IN YOUR DATASET")
print("-" * 60)
class_counts = df_distribution.groupby('taxonomic_class').agg(
    total_images=('species', 'count'),
    unique_species=('species', 'nunique')
).sort_values(by='total_images', ascending=False)

for t_class, row in class_counts.iterrows():
    print(f"🧬 Class: {t_class:<12} | 🌲 Unique Species: {row['unique_species']:<2} | 📸 Total Images: {row['total_images']}")
print("-" * 60)

# 5. Print Micro Roster Grouped cleanly by Class
print("\n🔍 DETAILED ROSTER GROUPED BY TAXONOMIC CLASS:")
grouped = df_distribution.groupby(['taxonomic_class', 'species']).size().to_frame('img_count')

current_class = ""
for (t_class, species_name), row in grouped.iterrows():
    if t_class != current_class:
        print(f"\n🟢 CLASS: {t_class.upper()}")
        print("=" * 60)
        current_class = t_class
    print(f"  🔹 {species_name:<35} (📸 {row['img_count']} images)")

📊 MACRO CLASS DISTRIBUTION IN YOUR DATASET
------------------------------------------------------------
🧬 Class: Aves         | 🌲 Unique Species: 15 | 📸 Total Images: 2287
🧬 Class: Reptilia     | 🌲 Unique Species: 7  | 📸 Total Images: 850
🧬 Class: Mammalia     | 🌲 Unique Species: 5  | 📸 Total Images: 500
🧬 Class: Amphibia     | 🌲 Unique Species: 3  | 📸 Total Images: 300
------------------------------------------------------------

🔍 DETAILED ROSTER GROUPED BY TAXONOMIC CLASS:

🟢 CLASS: AMPHIBIA
  🔹 Anaxyrus boreas                     (📸 100 images)
  🔹 Dicamptodon tenebrosus              (📸 100 images)
  🔹 Ensatina eschscholtzii              (📸 100 images)

🟢 CLASS: AVES
  🔹 Aechmophorus occidentalis           (📸 150 images)
  🔹 Aythya valisineria                  (📸 150 images)
  🔹 Bubo virginianus                    (📸 150 images)
  🔹 Buteo lineatus                      (📸 150 images)
  🔹 Calcarius lapponicus                (📸 150 images)
  🔹 Cathartes aura                      (📸 15

In [4]:
# # =====================================================================
# # BALANCED CALIFORNIA DATASET CREATOR (100-IMAGE CAP PER SPECIES)
# # =====================================================================
# import os
# import pandas as pd

# DOWNLOAD_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_images/extra_images'
# CSV_DIR = '/Users/alan/Desktop/spring_2026/ds3/WildScan/Data/WildScan_Data/animal_phylum_data/all_csvs'

# print("📁 Step 1: Scanning physical species folders with 100-image cap...")
# current_species_folders = [f for f in os.listdir(DOWNLOAD_DIR) if not f.startswith('.')]

# all_records = []
# for folder in current_species_folders:
#     species_name = folder.replace('_', ' ').strip()
#     folder_path = os.path.join(DOWNLOAD_DIR, folder)
#     images = [img for img in os.listdir(folder_path) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
#     # CAP: Slice the list to keep a maximum of 100 images for this species
#     capped_images = images[:100]
    
#     for img in capped_images:
#         all_records.append({
#             'species': species_name,
#             'local_path': os.path.join(folder_path, img)
#         })

# combined_df = pd.DataFrame(all_records)

# print("\n🌲 Step 2: Extracting California checklist from metadata sheets...")
# target_csvs = {
#     'Mammalia':  'mammal_class.csv',
#     'Amphibia':  'amphibia_class.csv',
#     'Reptilia':  'reptilia_class.csv',
#     'Aves':      'aves_class.csv',
#     'Insecta':   'insecta_class.csv',
#     'Arachnida': 'arachnida_class.csv',
# }

# ca_species_set = set()
# species_to_class_map = {}

# for class_name, csv_file in target_csvs.items():
#     csv_path = os.path.join(CSV_DIR, csv_file)
#     if os.path.exists(csv_path):
#         df_csv = pd.read_csv(csv_path, usecols=['scientific_name', 'place_admin1_name', 'place_state_name', 'place_guess'], low_memory=False)
        
#         # Build global species-to-class map for taxonomy printout
#         for s in df_csv['scientific_name'].dropna().unique():
#             species_to_class_map[str(s).strip().lower()] = class_name
            
#         # Filter for CA lines
#         ca_mask = (
#             (df_csv['place_admin1_name'].astype(str).str.strip() == 'California') |
#             (df_csv['place_state_name'].astype(str).str.strip().str.lower() == 'california') |
#             (df_csv['place_guess'].astype(str).str.lower().str.contains('california', na=False))
#         )
#         for s in df_csv[ca_mask]['scientific_name'].dropna().unique():
#             ca_species_set.add(str(s).strip().lower())

# print("\n⚡ Step 3: Filtering and mapping taxonomy...")
# # Keep rows matching California native checklist
# is_ca_native = combined_df['species'].str.lower().str.strip().isin(ca_species_set)
# combined_df = combined_df[is_ca_native].reset_index(drop=True)

# # Add taxonomic class mappings
# combined_df['taxonomic_class'] = combined_df['species'].str.lower().str.strip().map(species_to_class_map)
# combined_df['taxonomic_class'] = combined_df['taxonomic_class'].fillna('Other / Chordata')

# # Map clean categorical targets (0 to N-1) for PyTorch
# unique_ca_species = sorted(combined_df['species'].unique())
# species_to_id = {species: idx for idx, species in enumerate(unique_ca_species)}
# combined_df['label'] = combined_df['species'].map(species_to_id)

# print("\n📊 FINAL MACRO CLASS DISTRIBUTION")
# print("-" * 60)
# class_counts = combined_df.groupby('taxonomic_class').agg(
#     total_images=('species', 'count'),
#     unique_species=('species', 'nunique')
# ).sort_values(by='total_images', ascending=False)

# for t_class, row in class_counts.iterrows():
#     print(f"🧬 Class: {t_class:<12} | 🌲 Unique Species: {row['unique_species']:<2} | 📸 Total Capped Images: {row['total_images']}")
# print("-" * 60)

# print("\n🔍 FINALIZE TAXONOMIC BALANCED ROSTER:")
# grouped = combined_df.groupby(['taxonomic_class', 'species']).size().to_frame('img_count')

# current_class = ""
# for (t_class, species_name), row in grouped.iterrows():
#     if t_class != current_class:
#         print(f"\n🟢 CLASS: {t_class.upper()}")
#         print("=" * 60)
#         current_class = t_class
#     print(f"  🔹 {species_name:<35} (📸 {row['img_count']}/100 images)")

In [5]:
# =====================================================================
# CELL 3: STRATIFIED TRAIN/VAL SPLIT
# =====================================================================
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    combined_df, 
    test_size=0.2, 
    stratify=combined_df['label'], 
    random_state=42
)

print("🌲 Dataset Partitions Finalized!")
print(f"☀️ Training Pool:   {len(train_df)} images")
print(f"🌙 Validation Pool: {len(val_df)} images")
print(f"🎯 Unique Classes:  {train_df['label'].nunique()} (Successfully Synced)")

🌲 Dataset Partitions Finalized!
☀️ Training Pool:   3149 images
🌙 Validation Pool: 788 images
🎯 Unique Classes:  30 (Successfully Synced)


In [6]:
# =====================================================================
# CELL 4: DATASET WRAPPER AND AUGMENTATION PIPELINE
# =====================================================================
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class WildScanDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'local_path']
        label = int(self.df.loc[idx, 'label'])
        
        # Open image and convert to standard RGB
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# =====================================================================
# UPGRADED HIGH-VARIANCE AUGMENTATION PIPELINE
# =====================================================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 1. Instantiate Datasets (Created FIRST so they exist in memory)
train_dataset = WildScanDataset(train_df, transform=train_transform)
val_dataset = WildScanDataset(val_df, transform=val_transform)

# 2. Build DataLoaders (Switched num_workers=0 to play nice with MPS)
num_classes = len(unique_ca_species)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print("🔥 High-variance training augmentations successfully injected!")
print(f"🚀 DataLoaders ready. Training Batches: {len(train_loader)} | Validation Batches: {len(val_loader)}")

🔥 High-variance training augmentations successfully injected!
🚀 DataLoaders ready. Training Batches: 99 | Validation Batches: 25


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f"💻 EMERGENCY POLISHING ENGINE ACTIVATED. Target Device: {device}")

# 1. Recreate structure and load your 68.78% peak weights
num_classes = len(unique_ca_species)
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=num_classes)

checkpoint_path = 'production_wildscan_efficientnet_unfrozen.pth'
print(f"🔄 Loading peak checkpoint ({checkpoint_path})...")
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model = model.to(device)

# 2. Drop learning rates down to microscopic levels to prevent overfitting
# High weight decay (0.1) acts as an aggressive penalty to stop memorization
optimizer = optim.AdamW(model.parameters(), lr=5e-6, weight_decay=1e-1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.15) # Increased smoothing to fight noise

POLISH_EPOCHS = 5
best_val_acc = 68.78  # Start tracking from your current record

print(f"✨ Polishing decision boundaries across {POLISH_EPOCHS} micro-epochs...")
print("-" * 75)

for epoch in range(POLISH_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        if not isinstance(batch, (list, tuple)) or len(batch) != 2: continue
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    # Validation Pass
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            if not isinstance(batch, (list, tuple)) or len(batch) != 2: continue
            images, labels = batch
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    epoch_val_acc = (val_correct / max(val_total, 1)) * 100
    print(f"Polish Epoch [{epoch+1}/{POLISH_EPOCHS}] | Train Acc: {(correct/total)*100:.2f}% | 🔵 Val Acc: {epoch_val_acc:.2f}%")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'final_70_wildscan_model.pth')
        print(f"🎉 TARGET SHATTERED! New Peak: {best_val_acc:.2f}% Saved!")

print(f"\n🏁 Process complete. Highest locked accuracy: {best_val_acc:.2f}%")

/opt/anaconda3/envs/happiness_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


💻 CRITICAL ACCURACY PUSH. Targeted Device: mps
📦 Loading efficientnet_b0 for 30 optimized classes...
🔓 Unlocking entire network structure for full California species adaptation...
🚀 Launching Full-Adaptation EfficientNet Pipeline...
---------------------------------------------------------------------------
Epoch [01/30]
  🟢 Train Loss: 4.2460 | Train Acc: 7.05%
  🔵 Val Loss:   3.3090 | Val Acc:   13.71%
  -----------------------------------------------------------------
⭐ TARGET CELL BREAKTHROUGH: 13.71% Saved!
Epoch [02/30]
  🟢 Train Loss: 3.3791 | Train Acc: 16.64%
  🔵 Val Loss:   2.7952 | Val Acc:   28.17%
  -----------------------------------------------------------------
⭐ TARGET CELL BREAKTHROUGH: 28.17% Saved!
Epoch [03/30]
  🟢 Train Loss: 2.8833 | Train Acc: 26.80%
  🔵 Val Loss:   2.4585 | Val Acc:   38.71%
  -----------------------------------------------------------------
⭐ TARGET CELL BREAKTHROUGH: 38.71% Saved!
Epoch [04/30]
  🟢 Train Loss: 2.5669 | Train Acc: 36.14%
  🔵 V

In [8]:
import torch
import timm

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f"💻 Evaluating on device: {device}")

# 1. Load the architecture structural layout
num_classes = len(unique_ca_species)
model = timm.create_model('resnet18', pretrained=False, num_classes=num_classes)

# 2. Pull in your saved weights file
checkpoint_path = 'production_wildscan_resnet18.pth'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model = model.to(device)

# 3. Freeze the layers completely for testing
model.eval()

val_correct, val_total = 0, 0

print("🔬 Running inference pass across test stream...")
with torch.no_grad():
    for batch in val_loader:
        if not isinstance(batch, (list, tuple)) or len(batch) != 2:
            continue
            
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        val_total += labels.size(0)
        val_correct += predicted.eq(labels).sum().item()

final_acc = (val_correct / max(val_total, 1)) * 100
print("-" * 50)
print(f"🎯 FINAL TEST ACCURACY: {final_acc:.2f}%")
print("-" * 50)

💻 Evaluating on device: mps
🔬 Running inference pass across test stream...
--------------------------------------------------
🎯 FINAL TEST ACCURACY: 58.53%
--------------------------------------------------
